# Dataset Checks

Optional technical review notebook. The production CLI workflow lives in `../scripts/`.


# Notebook 05 | SPAM to FABLE reproducibility and publication preparation

This is the final release audit notebook for the historical crop-area series. It checks the versioned Notebook 04 candidate, independently repeats essential data checks, verifies the accepted MIRCA2000 corrections and records the reproducibility evidence needed for a public release.

It does not overwrite the Calculator's canonical FINAL. It also does not equate a previously saved QC report or a repeated read of an input file with a clean rebuild.

**Current release boundary:** Notebook 03 and Notebook 04 have passed their scientific audits. A complete source-to-release run in a fresh workspace, the final Notebook 03 source file, source-data licensing and explicit release approval remain separate gates until verified.

Run the cells in order with your `pythongeo` environment. The only configuration normally needed is the project root and the folder holding the final saved notebooks. No long separators or retired exploration cells are included.

## 01 | Paths and a safe release workspace

The candidate stays frozen under `04_final_release/publication_candidate`. This notebook writes only to `05_publication_audit`. A different version receives a separate folder.

In [ ]:
# 01 | Configuration and helpers
from pathlib import Path
import hashlib
import importlib.metadata
import json
import os
import platform
import shutil
import sys

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path(os.environ.get(
    "SPAM_FABLE_ROOT", r"DATA_ROOT\SPAM_FABLE_FINAL"
)).expanduser().resolve()
VERSION = "v1_0_candidate"
CANDIDATE_DIR = PROJECT_ROOT / "04_final_release" / "publication_candidate" / VERSION
AUDIT_DIR_05 = PROJECT_ROOT / "05_publication_audit" / VERSION
BUNDLE_DIR = AUDIT_DIR_05 / "publication_bundle"
NOTEBOOK_DIR = Path(os.environ.get("SPAM_NOTEBOOK_DIR", str(Path.cwd()))).expanduser().resolve()

# Metadata is deliberately not guessed. Complete these before public deposit.
PUBLICATION = {
    "title": "SPAM to FABLE historical crop-area and irrigation dataset, 2000 to 2020",
    "creators": [],  # [{"name": "Family, Given", "affiliation": "...", "orcid": "..."}]
    "data_license": "",  # A reviewed license identifier, for example CC-BY-4.0
    "code_license": "",  # A reviewed software license identifier
    "source_redistribution_cleared": False,
    "institutional_release_approved": False,
    "doi": "",  # Optional until a DOI is reserved or issued
}

# Do not silently package large spatial Parquet files or raw SPAM inputs.
INCLUDE_OPTIONAL_PARQUET = True
INCLUDE_SPATIAL_REPLAY = False
INCLUDE_MAPPED_GRIDS = False  # Optional large mapped 2000, 2005, 2010, 2020 and 2015 files

AREAS_05 = [
    "harvested_irrigated_ha", "harvested_rainfed_ha", "harvested_total_ha",
    "physical_irrigated_ha", "physical_rainfed_ha", "physical_total_ha",
]
PRIMARY_05 = [
    "harvested_irrigated_ha", "harvested_rainfed_ha",
    "physical_irrigated_ha", "physical_rainfed_ha",
]
KEYS_05 = ["country_iso3", "year", "fable_product"]
UNITS_05 = {
    "harvested_irrigated_ha": "irrigated_harvested_area_1000ha",
    "harvested_rainfed_ha": "rainfed_harvested_area_1000ha",
    "harvested_total_ha": "total_harvested_area_1000ha",
    "physical_irrigated_ha": "irrigated_physical_area_1000ha",
    "physical_rainfed_ha": "rainfed_physical_area_1000ha",
    "physical_total_ha": "total_physical_area_1000ha",
}
CALCULATOR_05 = [
    *KEYS_05, *AREAS_05,
    "harvesting_intensity_irrigated", "harvesting_intensity_rainfed",
    "harvesting_intensity_total", "intensity_status_irrigated",
    "intensity_status_rainfed", "intensity_status_total",
    "data_status", "temporal_status", "gapfill_method", "source_dataset",
    "source_country_coverage", "fable_area_available", "mapping_complete",
    "calculator_ready", "high_intensity_review_irrigated",
    "high_intensity_review_rainfed", "high_intensity_review_total",
    "high_intensity_review_any", "intensity_quality_flag",
    *UNITS_05.values(),
]
HA_TOL_05 = 1e-5
INT_TOL_05 = 1e-10
IDENTITY_AREA_TOL_05 = 1e-7

def sha256_05(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def json_05(path):
    with Path(path).open(encoding="utf-8") as stream:
        return json.load(stream)

def check_05(condition, message):
    if not bool(condition):
        raise AssertionError(message)

def numeric_05(series):
    return pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)

def bool_05(series):
    if pd.api.types.is_bool_dtype(series):
        v = series.astype("boolean")
    else:
        v = series.astype("string").str.lower().str.strip().map({
            "true": True, "false": False, "1": True, "0": False,
            "yes": True, "no": False,
        }).astype("boolean")
    check_05(v.notna().all(), f"Unrecognized boolean values: {series.name}")
    return v.to_numpy(dtype=bool)

def copy_checked_05(source, target):
    source, target = Path(source), Path(target)
    check_05(source.is_file(), f"Missing package input: {source}")
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_file():
        check_05(sha256_05(source) == sha256_05(target),
                 f"Existing package file differs. Use a new VERSION: {target}")
    else:
        shutil.copy2(source, target)
        check_05(sha256_05(source) == sha256_05(target),
                 f"File copy did not preserve the checksum: {target}")
    return target

AUDIT_DIR_05.mkdir(parents=True, exist_ok=True)
print("Candidate:", CANDIDATE_DIR)
print("Publication audit:", AUDIT_DIR_05)
print("Notebook source folder:", NOTEBOOK_DIR)
print("Canonical Calculator FINAL will not be written.")

## 02 | Freeze the accepted scientific evidence

Before making another package copy, confirm Notebook 03's eight gates and Notebook 04's scientific result. The historical candidate and the old canonical FINAL must still have the SHA-256 values recorded in their manifests.

In [ ]:
# 02 | Validate inputs and source checksums
REQUIRED_05 = {
    "notebook03_manifest": PROJECT_ROOT / "03_reconstruction_audit" / "03_lineage_manifest.json",
    "notebook04_manifest": CANDIDATE_DIR / "04_release_candidate_manifest.json",
    "candidate_csv": CANDIDATE_DIR / (
        "04_FABLE_C_SPAM_historical_2000_2020_MIRCA23_CORRECTED_" + VERSION + ".csv.gz"
    ),
    "candidate_parquet": CANDIDATE_DIR / (
        "04_FABLE_C_SPAM_historical_2000_2020_" + VERSION + ".parquet"
    ),
    "dictionary": CANDIDATE_DIR / "04_historical_data_dictionary.csv",
    "changes": CANDIDATE_DIR / "04_MIRCA2000_approved_23_change_register.csv",
    "release_summary": CANDIDATE_DIR / "qc" / "04_release_candidate_validation.csv",
    "numerical_qc": CANDIDATE_DIR / "qc" / "04_calculator_numerical_QC.csv",
    "year_coverage": CANDIDATE_DIR / "qc" / "04_historical_year_coverage.csv",
    "spatial_qc": CANDIDATE_DIR / "qc" / "04_2015_native_grid_replay_QC.csv",
    "country_qc": CANDIDATE_DIR / "qc" / "04_2015_country_product_summary.csv",
    "mirca_qc": CANDIDATE_DIR / "qc" / "04_MIRCA23_historical_QC.csv",
    "canonical_final": PROJECT_ROOT / "04_final_release" / "calculator" /
        "04_FABLE_C_SPAM_historical_input_2000_2020_FINAL.csv.gz",
    "notebook10c_summary": PROJECT_ROOT / "03_reconstruction_audit" /
        "10C_candidate_validation" / "10C_disk_validation_summary.csv",
}
missing_05 = {name: str(p) for name, p in REQUIRED_05.items() if not p.is_file()}
check_05(not missing_05, f"Missing required release evidence: {missing_05}")

manifest03_05 = json_05(REQUIRED_05["notebook03_manifest"])
manifest04_05 = json_05(REQUIRED_05["notebook04_manifest"])
summary04_05 = pd.read_csv(REQUIRED_05["release_summary"])
summary10c_05 = pd.read_csv(REQUIRED_05["notebook10c_summary"])

check_05(manifest03_05.get("audit_validated") is True, "Notebook 03 audit is not approved.")
check_05(manifest03_05.get("mandatory_audit_gates_passed") == 8, "Notebook 03 gate count changed.")
check_05(manifest03_05.get("mandatory_audit_gates_total") == 8, "Notebook 03 denominator changed.")
check_05(manifest04_05.get("scientific_validation_status") ==
         "PASS_RELEASE_CANDIDATE_SCIENTIFIC_VALIDATION", "Notebook 04 scientific status changed.")
check_05(len(summary04_05) == 1 and summary04_05.loc[0, "status"] ==
         "PASS_RELEASE_CANDIDATE_SCIENTIFIC_VALIDATION", "Notebook 04 summary is not PASS.")
check_05(len(summary10c_05) == 1 and summary10c_05.loc[0, "status"] ==
         "PASS_HISTORICAL_CANDIDATE_DISK_VALIDATION", "Notebook 03 disk QC is not PASS.")

candidate_hash_05 = sha256_05(REQUIRED_05["candidate_csv"])
canonical_hash_05 = sha256_05(REQUIRED_05["canonical_final"])
check_05(candidate_hash_05 == manifest04_05["historical_candidate_sha256"],
         "Candidate differs from the frozen Notebook 04 manifest.")
check_05(candidate_hash_05 == manifest03_05["historical_candidate_sha256"],
         "Candidate differs from the validated Notebook 03 file.")
check_05(canonical_hash_05 == manifest04_05["canonical_final_sha256"],
         "Original canonical FINAL changed since Notebook 04.")
check_05(canonical_hash_05 == manifest03_05["canonical_final_sha256"],
         "Original canonical FINAL changed since Notebook 03.")
check_05(manifest04_05.get("public_release_approved") is False,
         "Review the previously declared release boundary.")

qc04_05 = pd.read_csv(REQUIRED_05["numerical_qc"])
check_05(len(qc04_05) == 19 and qc04_05["failures"].eq(0).all(),
         "Notebook 04's 19 numerical checks are no longer all PASS.")

print("Notebook 03 scientific audit: 8 of 8 PASS")
print("Notebook 04 numerical checks: 19 of 19 PASS")
print("Historical candidate SHA-256:", candidate_hash_05)
print("Canonical FINAL SHA-256:", canonical_hash_05)

## 03 | Repeat the five-year data validation independently

Read the actual CSV, not the already exported QC counts. The accepted historical convention retains either zero or missing for zero harvested and zero physical area. It must never cap harvesting intensity at one.

In [ ]:
# 03 | Recheck the release data and the Calculator schema
historical_05 = pd.read_csv(REQUIRED_05["candidate_csv"], low_memory=False)
check_05(list(historical_05.columns) == CALCULATOR_05,
         "The release candidate does not contain the frozen 34-column schema.")
check_05(historical_05.shape == (52864, 34), "Historical panel dimensions changed.")
check_05(historical_05[KEYS_05].notna().all().all(), "Missing historical identifiers.")
check_05(not historical_05.duplicated(KEYS_05).any(), "Duplicate historical keys.")

historical_05["country_iso3"] = historical_05["country_iso3"].astype("string").str.upper().str.strip()
historical_05["fable_product"] = historical_05["fable_product"].astype("string").str.lower().str.strip()
historical_05["year"] = pd.to_numeric(historical_05["year"], errors="raise").astype(int)

by_year_05 = (historical_05.groupby("year", as_index=False)
    .agg(countries=("country_iso3", "nunique"),
         products=("fable_product", "nunique"), records=("fable_product", "size"))
    .sort_values("year"))
expected_panel_05 = {2000: (177, 9912), 2005: (184, 10304), 2010: (201, 11256),
                     2015: (203, 11368), 2020: (179, 10024)}
for rec in by_year_05.itertuples(index=False):
    check_05((rec.countries, rec.records) == expected_panel_05[rec.year],
             f"Unexpected country and product coverage in {rec.year}.")
    check_05(rec.products == 56, f"Expected 56 products for {rec.year}.")
check_05(set(by_year_05.year) == set(expected_panel_05), "Unexpected historical years.")
check_05(historical_05.groupby(["year", "country_iso3"])["fable_product"].nunique().eq(56).all(),
         "Incomplete 56-product country panel.")
products_05 = set(historical_05.loc[historical_05.year.eq(2000), "fable_product"])
check_05(all(set(x.fable_product) == products_05 for _, x in historical_05.groupby("year")),
         "FABLE product universe changed between years.")

# Verify each original numerical rule directly from the saved data.
checks_05 = []
for group in ["harvested", "physical"]:
    ir, rf, total = [numeric_05(historical_05[f"{group}_{part}_ha"])
                     for part in ["irrigated", "rainfed", "total"]]
    available = np.isfinite(ir) & np.isfinite(rf) & np.isfinite(total)
    checks_05 += [
        (f"{group}_missing_components", int((~available).sum())),
        (f"{group}_negative_area", int((np.column_stack([ir, rf, total]) < -HA_TOL_05).any(axis=1).sum())),
        (f"{group}_sum_identity", int((available & (np.abs(ir + rf - total) > HA_TOL_05)).sum())),
    ]

for part in ["irrigated", "rainfed", "total"]:
    h = numeric_05(historical_05[f"harvested_{part}_ha"])
    p = numeric_05(historical_05[f"physical_{part}_ha"])
    saved = numeric_05(historical_05[f"harvesting_intensity_{part}"])
    expected = np.full(len(h), np.nan)
    positive_p = np.isfinite(h) & np.isfinite(p) & (p > 0)
    zero_zero = np.isfinite(h) & np.isfinite(p) & (h == 0) & (p == 0)
    expected[positive_p] = h[positive_p] / p[positive_p]
    expected[zero_zero] = 0.0
    valid = np.isclose(saved, expected, atol=INT_TOL_05, rtol=1e-12, equal_nan=True)
    valid |= zero_zero & np.isnan(saved)
    review = bool_05(historical_05[f"high_intensity_review_{part}"])
    checks_05 += [
        (f"intensity_{part}", int((~valid).sum())),
        (f"high_intensity_flag_{part}", int((review != (saved >= 5)).sum())),
    ]

flags_05 = np.column_stack([bool_05(historical_05[f"high_intensity_review_{p}"])
                            for p in ["irrigated", "rainfed", "total"]])
checks_05.append(("high_intensity_flag_any", int((
    bool_05(historical_05["high_intensity_review_any"]) != flags_05.any(axis=1)).sum())))
for area, in_thousand in UNITS_05.items():
    checks_05.append((f"units_{in_thousand}", int((~np.isclose(
        numeric_05(historical_05[area]), numeric_05(historical_05[in_thousand]) * 1000,
        atol=HA_TOL_05, rtol=0, equal_nan=True)).sum())))
checks_05 += [
    ("calculator_ready", int((~bool_05(historical_05["calculator_ready"])).sum())),
    ("country_coverage", int((~bool_05(historical_05["source_country_coverage"])).sum())),
]
rechecked_qc_05 = pd.DataFrame(checks_05, columns=["check", "failures"])
rechecked_qc_05["status"] = np.where(rechecked_qc_05.failures.eq(0), "PASS", "FAIL")
check_05(rechecked_qc_05.failures.eq(0).all(), "Independent numerical QC found a discrepancy.")
print("Five-year country and product panel: PASS")
print("Independent numerical checks:", len(rechecked_qc_05), "of", len(rechecked_qc_05), "PASS")
display(by_year_05)

## 04 | Check the original historical records and the 23 approved corrections

Only the selected 2000 irrigation-share records may have changed. The remaining records must match the original Calculator table within the already accepted CSV read/write precision. Check the before and after correction register rather than accepting a summary count.

In [ ]:
# 04 | Independent historical identity and MIRCA check
original_05 = pd.read_csv(REQUIRED_05["canonical_final"], low_memory=False)
changes_05 = pd.read_csv(REQUIRED_05["changes"], low_memory=False)
check_05(list(original_05.columns) == CALCULATOR_05 and len(original_05) == len(historical_05),
         "The canonical historical schema or row count is inconsistent.")

def key_index_05(frame):
    keys = frame[KEYS_05].copy()
    keys["country_iso3"] = keys["country_iso3"].astype("string").str.upper().str.strip()
    keys["fable_product"] = keys["fable_product"].astype("string").str.lower().str.strip()
    keys["year"] = pd.to_numeric(keys["year"], errors="raise").astype(int)
    idx = pd.MultiIndex.from_frame(keys)
    check_05(idx.is_unique and not keys.isna().any().any(), "Invalid historical or correction keys.")
    return idx

candidate_keys_05 = key_index_05(historical_05)
original_keys_05 = key_index_05(original_05)
changes_keys_05 = key_index_05(changes_05)
check_05(candidate_keys_05.equals(original_keys_05), "Original and candidate record order differs.")
check_05(len(changes_keys_05) == 23 and (changes_keys_05.get_level_values("year") == 2000).all(),
         "Change register must contain exactly 23 year-2000 records.")
change_pos_05 = candidate_keys_05.get_indexer(changes_keys_05)
check_05((change_pos_05 >= 0).all(), "A MIRCA change key is missing from the release.")
selected_05 = np.zeros(len(historical_05), dtype=bool)
selected_05[change_pos_05] = True

identity_fail_05 = {}
max_non_target_ha_diff_05 = 0.0
for col in CALCULATOR_05:
    a = original_05.loc[~selected_05, col].reset_index(drop=True)
    b = historical_05.loc[~selected_05, col].reset_index(drop=True)
    if pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b):
        tol = (IDENTITY_AREA_TOL_05 if col in AREAS_05
               else IDENTITY_AREA_TOL_05 / 1000 if col in UNITS_05.values()
               else INT_TOL_05 if col.startswith("harvesting_intensity_") else 0)
        av, bv = numeric_05(a), numeric_05(b)
        diff = np.abs(av - bv)
        if col in AREAS_05 and np.isfinite(diff).any():
            max_non_target_ha_diff_05 = max(max_non_target_ha_diff_05,
                                            float(np.nanmax(diff[np.isfinite(diff)])))
        failed = int((~np.isclose(av, bv, atol=tol, rtol=0, equal_nan=True)).sum())
    else:
        failed = int((~a.astype("string").fillna("<NULL>").eq(
            b.astype("string").fillna("<NULL>"))).sum())
    if failed:
        identity_fail_05[col] = failed
check_05(not identity_fail_05, f"Non-target records changed: {identity_fail_05}")

for col in AREAS_05:
    check_05(f"before_{col}" in changes_05 and f"after_{col}" in changes_05,
             f"Missing correction evidence for {col}.")
    original_values = numeric_05(original_05.iloc[change_pos_05][col])
    current_values = numeric_05(historical_05.iloc[change_pos_05][col])
    before_values = numeric_05(changes_05[f"before_{col}"])
    after_values = numeric_05(changes_05[f"after_{col}"])
    check_05(np.isclose(before_values, original_values, atol=HA_TOL_05, rtol=0).all(),
             f"Before-correction evidence differs for {col}.")
    check_05(np.isclose(after_values, current_values, atol=HA_TOL_05, rtol=0).all(),
             f"Approved MIRCA corrections differ for {col}.")
for group in ["harvested", "physical"]:
    before = numeric_05(original_05.iloc[change_pos_05][f"{group}_total_ha"])
    after = numeric_05(historical_05.iloc[change_pos_05][f"{group}_total_ha"])
    check_05(np.isclose(before, after, atol=HA_TOL_05, rtol=0).all(),
             f"The correction changed {group} total area.")

baseline_identity_05 = pd.DataFrame([{
    "non_target_records_checked": int((~selected_05).sum()),
    "approved_corrections_verified": len(changes_05),
    "max_non_target_area_difference_ha": max_non_target_ha_diff_05,
    "failed_non_target_fields": len(identity_fail_05),
    "status": "PASS_HISTORICAL_IDENTITY_AND_MIRCA23",
}])
display(baseline_identity_05)

## 05 | Confirm the independently replayed 2015 geography and interpolation

Notebook 04 already repeated the 2015 spatial interpolation at 5 arc-minute resolution. Verify both persisted QC files, the 2015 country totals and the reference hashes. This remains an independently validated spatial replay, not a fresh raw-source rebuild.

In [ ]:
# 05 | Recheck the spatial evidence and locked source hashes
spatial_qc_05 = pd.read_csv(REQUIRED_05["spatial_qc"])
country_qc_05 = pd.read_csv(REQUIRED_05["country_qc"])
mirca_qc_05 = pd.read_csv(REQUIRED_05["mirca_qc"])
for name, frame in [("spatial", spatial_qc_05), ("country", country_qc_05),
                    ("MIRCA2000", mirca_qc_05)]:
    check_05(len(frame) == 1 and "status" in frame.columns, f"Invalid {name} QC summary.")
check_05(spatial_qc_05.loc[0, "status"] == "PASS_2015_NATIVE_GRID_COMPONENT_REPLAY",
         "Saved 2015 grid replay did not PASS.")
check_05(country_qc_05.loc[0, "status"] == "PASS_2015_COUNTRY_CALCULATOR_RECONCILIATION",
         "Saved 2015 national reconciliation did not PASS.")
check_05(mirca_qc_05.loc[0, "status"] == "PASS_2000_MIRCA23_CORRECTION_IN_RELEASE_CANDIDATE",
         "Saved MIRCA correction check did not PASS.")
check_05(int(spatial_qc_05.loc[0, "missing_keys"]) == 0 and
         int(spatial_qc_05.loc[0, "country_mismatches"]) == 0 and
         int(spatial_qc_05.loc[0, "component_mismatches"]) == 0,
         "2015 grid reconciliation has mismatches.")
check_05(int(country_qc_05.loc[0, "mismatch_records"]) == 0 and
         int(mirca_qc_05.loc[0, "approved_area_failures"]) == 0 and
         int(mirca_qc_05.loc[0, "total_area_change_failures"]) == 0,
         "2015 country or 2000 MIRCA reconciliation has mismatches.")

# The 04 manifest records all 13 frozen input hashes. Check the
# current source file if it is still available at its recorded location.
source_manifest_05 = pd.read_csv(
    CANDIDATE_DIR / "qc" / "04_source_SHA256_inventory.csv", low_memory=False
)
check_05({"asset", "path", "sha256"}.issubset(source_manifest_05.columns),
         "Notebook 04's source fingerprint table is incomplete.")
changed_sources_05 = []
for row in source_manifest_05.itertuples(index=False):
    path = Path(row.path)
    if not path.is_file() or sha256_05(path) != row.sha256:
        changed_sources_05.append(row.asset)
check_05(not changed_sources_05, f"Missing or changed frozen inputs: {changed_sources_05}")
print("2015 spatial replay: PASS")
print("2015 national reconciliation: PASS")
print("Approved MIRCA2000 corrections: PASS")
print("Frozen source hashes:", len(source_manifest_05), "verified")

## 05B | Inventory the native and mapped spatial products

The final country-level Calculator CSV is the main versioned dataset. The native SPAM and mapped FABLE 5 arc-minute layers are also an important part of the processing record. Inventory them explicitly, even when they are too large or not licensed for inclusion in the public package. Large mapped layers can be added to a separate data archive after checking distribution rights.

In [ ]:
# 05B | Five-year spatial output inventory
SPATIAL_PRODUCTS_05 = {}
for year in [2000, 2005, 2010, 2020]:
    SPATIAL_PRODUCTS_05[f"native_{year}"] = (
        PROJECT_ROOT / "01_native_5arcmin" /
        f"01_SPAM_native_5arcmin_{year}.parquet"
    )
    SPATIAL_PRODUCTS_05[f"mapped_{year}"] = (
        PROJECT_ROOT / "03_observed_FABLE_5arcmin" /
        f"03_SPAM_FABLE_observed_5arcmin_{year}.parquet"
    )
SPATIAL_PRODUCTS_05["mapped_2015"] = (
    PROJECT_ROOT / "04_final_release" / "spatial_5arcmin" /
    "04_SPAM_FABLE_5arcmin_2015_interpolated.parquet"
)
spatial_inventory_rows_05 = []
for label, p in SPATIAL_PRODUCTS_05.items():
    spatial_inventory_rows_05.append({
        "asset": label,
        "path": str(p),
        "exists": p.is_file(),
        "bytes": p.stat().st_size if p.is_file() else pd.NA,
        "sha256": sha256_05(p) if p.is_file() else pd.NA,
        "role": "native_input_layer" if label.startswith("native_")
                else "mapped_spatial_intermediate",
        "packaged_by_default": False,
    })
spatial_inventory_05 = pd.DataFrame(spatial_inventory_rows_05)
print("Native and mapped spatial files available:",
      int(spatial_inventory_05.exists.sum()), "/", len(spatial_inventory_05))
display(spatial_inventory_05[["asset", "exists", "bytes", "role"]])

## 06 | Identify the exact source notebooks

A saved QC file is not a substitute for runnable source code. The final Notebook 03 is especially important because the earlier review file did not contain the subsequent 09B replay, 10B integration, 10C disk QC and final Cell 11. If the correct edited notebook is not saved in the configured folder, keep the source-code gate pending. Do not package a stale Notebook 03 file by mistake.

In [ ]:
# 06 | Resolve and review the publication notebook files
NOTEBOOK_CANDIDATES_05 = {
    "01": ["01_SPAM_Native_5arcmin_Publication_Review_v1.ipynb"],
    "02": ["02_SPAM_FABLE_Mapping_Publication_Review_v2.ipynb"],
    "03": ["03_SPAM2000_Reconstruction_Lineage_Audit_Publication_Review_v2.ipynb"],
    "04": ["04_SPAM_FABLE_Historical_Finalization_Publication_Candidate_FINAL.ipynb"],
    "05": ["05_SPAM_FABLE_Reproducibility_and_Publication_FINAL.ipynb"],
}

# You can replace an entry with your actual final notebook filename.
# The file must be the saved notebook containing your executed final cells.
NOTEBOOK_OVERRIDES_05 = {
    "01": "", "02": "", "03": "", "04": "", "05": "",
}

notebook_records_05 = []
notebook_paths_05 = {}
for step, names in NOTEBOOK_CANDIDATES_05.items():
    selected_name = NOTEBOOK_OVERRIDES_05.get(step) or names[0]
    possible = [NOTEBOOK_DIR / selected_name,
                NOTEBOOK_DIR / "notebooks" / selected_name]
    selected = next((p for p in possible if p.is_file()), None)
    present = selected is not None
    contains_final_03 = False
    if step == "03" and present:
        # Check the actual saved code, not cached output or notebook title.
        import nbformat
        raw_nb = nbformat.read(selected, as_version=4)
        source = "\n".join(c.source for c in raw_nb.cells if c.cell_type == "code")
        needed_tokens = [
            "PASS_OOIL_REPLAY_AND_SAVED_OUTPUT_CONSISTENCY",
            "PASS_HISTORICAL_CANDIDATE_DISK_VALIDATION",
            "03_lineage_manifest.json",
            "10B_FABLE_C_SPAM_historical_2000_2020_MIRCA23_RESTORED_CANDIDATE",
        ]
        contains_final_03 = all(t in source for t in needed_tokens)
    if present:
        notebook_paths_05[step] = selected
    notebook_records_05.append({
        "stage": step,
        "file": str(selected) if selected else str(possible[0]),
        "available": present,
        "final_03_code_verified": (contains_final_03 if step == "03" else pd.NA),
        "sha256": sha256_05(selected) if present else pd.NA,
    })

notebook_inventory_05 = pd.DataFrame(notebook_records_05)
source_code_complete_05 = bool(
    notebook_inventory_05.available.all()
    and notebook_inventory_05.loc[notebook_inventory_05.stage.eq("03"),
                                  "final_03_code_verified"].iloc[0] is True
)
# pandas may return numpy.bool_ rather than the Python singleton.
source_code_complete_05 = bool(
    notebook_inventory_05.available.all()
    and bool(notebook_inventory_05.loc[notebook_inventory_05.stage.eq("03"),
                                     "final_03_code_verified"].iloc[0])
)
print("Final saved notebook files verified:", source_code_complete_05)
display(notebook_inventory_05)
if not source_code_complete_05:
    print("Source-code package remains pending. Update NOTEBOOK_DIR or NOTEBOOK_OVERRIDES_05.")

## 07 | Evaluate real clean-start reproducibility

A true clean-start proof must execute source preparation, mapping, observed grid construction, 2000 magnitude completion and irrigation-share correction, 2015 interpolation and final assembly in a separate workspace. The result must be compared against this validated candidate. An old QC report, rerunning only Notebook 04 or copying the candidate to a new folder is not sufficient.

This cell accepts a separately executed run manifest only if its step records, source files, checksums, run logs and newly produced final dataset can all be checked. Until that evidence exists, the result stays pending.

In [ ]:
# 07 | Full clean-start evidence gate
CLEAN_MANIFEST_05 = AUDIT_DIR_05 / "clean_run" / "05_full_clean_run_manifest.json"
REQUIRED_BUILD_STAGES_05 = [
    "01_native_source_preparation", "02_frozen_mapping",
    "03_observed_grid_mapping", "03_2000_country_completion",
    "04_2015_grid_interpolation", "04_final_historical_assembly",
]

def compare_fresh_historical_05(path):
    fresh = pd.read_csv(path, low_memory=False)
    if list(fresh.columns) != CALCULATOR_05 or fresh.shape != historical_05.shape:
        return False, {"schema_or_shape": 1}
    fresh_idx = key_index_05(fresh)
    ref_idx = key_index_05(historical_05)
    if not fresh_idx.is_unique or not fresh_idx.equals(ref_idx):
        # Row order may differ in a real rebuild. Reorder before comparing.
        pos = fresh_idx.get_indexer(ref_idx)
        if (pos < 0).any() or len(fresh_idx) != len(ref_idx):
            return False, {"keys": 1}
        fresh = fresh.iloc[pos].reset_index(drop=True)
    result = {}
    for col in CALCULATOR_05:
        a = historical_05[col].reset_index(drop=True)
        b = fresh[col].reset_index(drop=True)
        if pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b):
            tolerance = (HA_TOL_05 if col in AREAS_05
                         else HA_TOL_05 / 1000 if col in UNITS_05.values()
                         else INT_TOL_05 if col.startswith("harvesting_intensity_") else 0.0)
            errors = int((~np.isclose(numeric_05(a), numeric_05(b),
                                      atol=tolerance, rtol=0, equal_nan=True)).sum())
        else:
            errors = int((~a.astype("string").fillna("<NULL>").eq(
                b.astype("string").fillna("<NULL>"))).sum())
        if errors:
            result[col] = errors
    return not result, result

clean_result_05 = {
    "status": "PENDING_FULL_CLEAN_SOURCE_TO_RELEASE_REBUILD",
    "stages_verified": 0,
    "stages_required": len(REQUIRED_BUILD_STAGES_05),
    "fresh_records_compared": 0,
    "difference_fields": None,
    "manifest_available": CLEAN_MANIFEST_05.is_file(),
}

if CLEAN_MANIFEST_05.is_file():
    clean_manifest = json_05(CLEAN_MANIFEST_05)
    execution_root = Path(clean_manifest.get("working_root", ""))
    check_05(execution_root.is_dir(), "Clean run workspace is missing.")
    check_05(execution_root.resolve() != PROJECT_ROOT.resolve(),
             "The original data root cannot be the clean workspace.")
    check_05(not str(execution_root.resolve()).startswith(str(CANDIDATE_DIR.resolve())),
             "Clean run must not use the publication candidate folder as output.")
    check_05(clean_manifest.get("execution_mode") == "fresh_isolated_raw_to_release",
             "Manifest is not a full raw-to-release execution record.")
    check_05(clean_manifest.get("reused_existing_processed_outputs") is False,
             "Processed outputs were reused in the supposed clean rebuild.")
    steps = {x.get("stage"): x for x in clean_manifest.get("steps", [])}
    for stage in REQUIRED_BUILD_STAGES_05:
        check_05(stage in steps, f"Missing required clean-rebuild stage: {stage}")
        item = steps[stage]
        check_05(item.get("executed") is True and item.get("exit_code") == 0,
                 f"Stage was not executed successfully: {stage}")
        code_path = Path(item["source_code"])
        log_path = Path(item["log_file"])
        check_05(code_path.is_file() and log_path.is_file(), f"Missing source or log for {stage}")
        check_05(sha256_05(code_path) == item["source_sha256"],
                 f"Source code fingerprint changed: {stage}")
        outputs = item.get("outputs", [])
        check_05(bool(outputs), f"Stage has no output artifacts: {stage}")
        for asset in outputs:
            output_path = Path(asset["path"])
            check_05(output_path.is_file() and sha256_05(output_path) == asset["sha256"],
                     f"Clean run output hash differs: {output_path}")
            check_05(execution_root.resolve() in output_path.resolve().parents,
                     f"Clean-run output is outside isolated workspace: {output_path}")
    candidate_path = Path(clean_manifest["final_historical_output"])
    check_05(execution_root.resolve() in candidate_path.resolve().parents,
             "Fresh final output must be inside the clean workspace.")
    check_05(candidate_path.is_file(), "Fresh historical dataset was not generated.")
    matching, difference_fields = compare_fresh_historical_05(candidate_path)
    clean_result_05.update({
        "status": ("PASS_FULL_CLEAN_SOURCE_TO_RELEASE_REBUILD" if matching
                   else "FAIL_FRESH_REBUILD_DATA_MISMATCH"),
        "stages_verified": len(REQUIRED_BUILD_STAGES_05),
        "fresh_records_compared": len(historical_05),
        "difference_fields": difference_fields,
    })

fresh_rebuild_pass_05 = (clean_result_05["status"] ==
                         "PASS_FULL_CLEAN_SOURCE_TO_RELEASE_REBUILD")
clean_summary_05 = pd.DataFrame([clean_result_05])
display(clean_summary_05)
if not CLEAN_MANIFEST_05.is_file():
    print("A full fresh rebuild has not been provided. The public-release gate stays PENDING.")

## 08 | Record packaging metadata and distribution permissions

A release must name its creators, licenses and redistribution permissions. The raw SPAM, FAOSTAT and MIRCA materials should not be copied to a public bundle without checking their individual terms. Empty metadata is a release blocker, not something to fill with an invented author or license.

In [ ]:
# 08 | Metadata and distribution review
metadata_05 = dict(PUBLICATION)
metadata_05["dataset_version"] = VERSION
metadata_05["historical_candidate_sha256"] = candidate_hash_05
metadata_05["countries_by_year"] = by_year_05.to_dict(orient="records")

metadata_complete_05 = bool(
    str(metadata_05.get("title", "")).strip()
    and isinstance(metadata_05.get("creators"), list)
    and len(metadata_05["creators"]) > 0
    and all(bool(str(c.get("name", "")).strip()) for c in metadata_05["creators"])
    and str(metadata_05.get("data_license", "")).strip()
    and str(metadata_05.get("code_license", "")).strip()
    and metadata_05.get("source_redistribution_cleared") is True
)
institutional_approval_05 = bool(metadata_05.get("institutional_release_approved") is True)

metadata_status_05 = pd.DataFrame([
    {"requirement": "Creator names and affiliations", "complete": bool(metadata_05["creators"]),
     "note": "List all approved creators in publication metadata."},
    {"requirement": "Dataset license", "complete": bool(metadata_05["data_license"]),
     "note": "Confirm a license that is compatible with input terms."},
    {"requirement": "Software license", "complete": bool(metadata_05["code_license"]),
     "note": "Check institutional code ownership before choosing a license."},
    {"requirement": "Source redistribution review", "complete": bool(metadata_05["source_redistribution_cleared"]),
     "note": "Do not redistribute restricted input datasets."},
    {"requirement": "Institutional release review", "complete": institutional_approval_05,
     "note": "A reviewed public release is separate from scientific QC."},
])
display(metadata_status_05)

## 09 | Assemble a review bundle without touching canonical production files

Create one small, coherent publication folder. Include the historical CSV, optional Parquet, the corrected-record register, data dictionary, the QC summaries, published notebook sources when verified and clear methods documentation. Do not automatically include raw SPAM inputs or the large spatial replay.

In [ ]:
# 09 | Build the versioned review package
for subdir in ["data", "qc", "notebooks", "docs", "reproducibility"]:
    (BUNDLE_DIR / subdir).mkdir(parents=True, exist_ok=True)

package_files_05 = {}
for name in ["candidate_csv", "dictionary", "changes"]:
    src = REQUIRED_05[name]
    package_files_05[name] = copy_checked_05(src, BUNDLE_DIR / "data" / src.name)
if INCLUDE_OPTIONAL_PARQUET:
    src = REQUIRED_05["candidate_parquet"]
    package_files_05["candidate_parquet"] = copy_checked_05(
        src, BUNDLE_DIR / "data" / src.name)

for name in ["release_summary", "numerical_qc", "year_coverage", "spatial_qc",
             "country_qc", "mirca_qc", "notebook10c_summary"]:
    src = REQUIRED_05[name]
    package_files_05[name] = copy_checked_05(src, BUNDLE_DIR / "qc" / src.name)
for name in ["notebook03_manifest", "notebook04_manifest"]:
    src = REQUIRED_05[name]
    package_files_05[name] = copy_checked_05(src, BUNDLE_DIR / "qc" / src.name)

# Copy all additional Notebook 03 gate and 2015 comparison tables for traceability.
for label, folder, pattern in [
    ("notebook03", PROJECT_ROOT / "03_reconstruction_audit", "03_*.csv"),
    ("notebook04", CANDIDATE_DIR / "qc", "04_*.csv"),
]:
    for src in sorted(folder.glob(pattern)):
        dest = BUNDLE_DIR / "qc" / label / src.name
        package_files_05[f"{label}/{src.name}"] = copy_checked_05(src, dest)

# Source notebooks enter the bundle only when all five final sources are verified.
if source_code_complete_05:
    for step, src in notebook_paths_05.items():
        package_files_05[f"notebook_{step}"] = copy_checked_05(
            src, BUNDLE_DIR / "notebooks" / src.name)
else:
    print("Notebooks not copied: final Notebook 03 source or another code file is missing.")

if INCLUDE_SPATIAL_REPLAY:
    large_src = (CANDIDATE_DIR / "spatial_replay" /
                 "04_2015_native_grid_component_midpoint_REPLAY.parquet")
    package_files_05["spatial_2015"] = copy_checked_05(
        large_src, BUNDLE_DIR / "data" / "spatial" / large_src.name)

rechecked_qc_05.to_csv(BUNDLE_DIR / "qc" / "05_independent_numerical_QC.csv", index=False)
by_year_05.to_csv(BUNDLE_DIR / "qc" / "05_historical_coverage.csv", index=False)
baseline_identity_05.to_csv(BUNDLE_DIR / "qc" / "05_MIRCA_and_baseline_identity.csv", index=False)
clean_summary_05.to_csv(BUNDLE_DIR / "qc" / "05_clean_rebuild_gate.csv", index=False)
notebook_inventory_05.to_csv(BUNDLE_DIR / "qc" / "05_source_notebook_inventory.csv", index=False)
metadata_status_05.to_csv(BUNDLE_DIR / "qc" / "05_metadata_review.csv", index=False)
spatial_inventory_05.to_csv(BUNDLE_DIR / "qc" / "05_spatial_lineage_inventory.csv", index=False)

if INCLUDE_MAPPED_GRIDS:
    check_05(metadata_05["source_redistribution_cleared"] is True,
             "Review distribution permissions before packaging large spatial layers.")
    for name, spatial_path in SPATIAL_PRODUCTS_05.items():
        if name.startswith("mapped_"):
            package_files_05[name] = copy_checked_05(
                spatial_path, BUNDLE_DIR / "data" / "mapped_5arcmin" / spatial_path.name)

# Include the reproducible source lock, but not copyrighted input files.
source_manifest_05.to_csv(BUNDLE_DIR / "reproducibility" /
                         "04_frozen_input_SHA256_inventory.csv", index=False)
(BUNDLE_DIR / "reproducibility" / "publication_metadata.template.json").write_text(
    json.dumps(metadata_05, indent=2, ensure_ascii=False), encoding="utf-8")

# Prepare an editable clean-start execution recipe. The four downstream
# processing entrypoints remain unrunnable until their real source code
# is parameterized for a fresh workspace. Never fabricate a PASS.
WORK_ROOT_05 = PROJECT_ROOT.parents[1]
REBUILD_RECIPE_05 = {
    "reference_project_root": str(WORK_ROOT_05),
    "working_root": str(AUDIT_DIR_05 / "clean_run" / "work"),
    "kernel": "pythongeo",
    "raw_inputs": [
        {"from": str(WORK_ROOT_05 / "Data" / "SPAM" / "CSV"), "to": "Data/SPAM/CSV"},
        {"from": str(WORK_ROOT_05 / "Data" / "mapping_audit"), "to": "Data/mapping_audit"},
        {"from": str(WORK_ROOT_05 / "Data" / "productlistfinal.xlsx"), "to": "Data/productlistfinal.xlsx"},
        {"from": str(PROJECT_ROOT / "04_final_release" / "qc" /
                     "04_grid_to_country_2015_FINAL.parquet"),
         "to": "Data/SPAM_FABLE_FINAL/04_final_release/qc/04_grid_to_country_2015_FINAL.parquet"},
    ],
    "steps": [
        {"stage": "01_native_source_preparation",
         "source_code": str(notebook_paths_05.get("01", "REPLACE_WITH_FINAL_NOTEBOOK01")),
         "runnable": "01" in notebook_paths_05,
         "expected_outputs": [
             "Data/SPAM_FABLE_FINAL/01_native_5arcmin/01_SPAM_native_5arcmin_2000.parquet",
             "Data/SPAM_FABLE_FINAL/01_native_5arcmin/01_SPAM_native_5arcmin_2010.parquet",
             "Data/SPAM_FABLE_FINAL/01_native_5arcmin/01_SPAM_native_5arcmin_2020.parquet"]},
        {"stage": "02_frozen_mapping",
         "source_code": str(notebook_paths_05.get("02", "REPLACE_WITH_FINAL_NOTEBOOK02")),
         "runnable": "02" in notebook_paths_05,
         "expected_outputs": [
             "Data/SPAM_FABLE_FINAL/02_mapping/02_SPAM_FABLE_direct_edges.csv",
             "Data/SPAM_FABLE_FINAL/02_mapping/02_SPAM_FABLE_operational_country_year_weights.csv.gz"]},
        {"stage": "03_observed_grid_mapping",
         "source_code": "REPLACE_WITH_PARAMETERIZED_OBSERVED_GRID_PRODUCTION_SOURCE",
         "runnable": False,
         "expected_outputs": [
             "Data/SPAM_FABLE_FINAL/03_observed_FABLE_5arcmin/03_SPAM_FABLE_observed_5arcmin_2010.parquet",
             "Data/SPAM_FABLE_FINAL/03_observed_FABLE_5arcmin/03_SPAM_FABLE_observed_5arcmin_2020.parquet"]},
        {"stage": "03_2000_country_completion",
         "source_code": "REPLACE_WITH_REPRODUCIBLE_2000_COMPLETION_AND_MIRCA_CORRECTION_SOURCE",
         "runnable": False,
         "expected_outputs": ["Data/SPAM_FABLE_FINAL/04_final_release/rebuilt_2000/2000_corrected.csv.gz"]},
        {"stage": "04_2015_grid_interpolation",
         "source_code": "REPLACE_WITH_PARAMETERIZED_2015_INTERPOLATION_SOURCE",
         "runnable": False,
         "expected_outputs": [
             "Data/SPAM_FABLE_FINAL/04_final_release/spatial_5arcmin/04_SPAM_FABLE_5arcmin_2015_interpolated.parquet"]},
        {"stage": "04_final_historical_assembly",
         "source_code": "REPLACE_WITH_REPRODUCIBLE_HISTORICAL_ASSEMBLY_SOURCE",
         "runnable": False,
         "expected_outputs": ["Data/SPAM_FABLE_FINAL/04_final_release/clean_rebuilt_historical_2000_2020.csv.gz"]},
    ],
    "final_historical_output": "Data/SPAM_FABLE_FINAL/04_final_release/clean_rebuilt_historical_2000_2020.csv.gz",
}
recipe_file_05 = BUNDLE_DIR / "reproducibility" / "05_full_clean_start_recipe.template.json"
if not recipe_file_05.exists():
    recipe_file_05.write_text(json.dumps(REBUILD_RECIPE_05, indent=2), encoding="utf-8")

runner_source_05 = NOTEBOOK_DIR / "05_SPAM_FABLE_clean_start_runner.py"
if runner_source_05.is_file():
    copy_checked_05(runner_source_05, BUNDLE_DIR / "reproducibility" / runner_source_05.name)
else:
    print("Optional clean-start runner not in NOTEBOOK_DIR. Download it separately.")

print("Review bundle created:", BUNDLE_DIR)
print("Primary candidate checksum retained:",
      sha256_05(package_files_05["candidate_csv"]) == candidate_hash_05)

## 10 | Write the methodology, usage and reproducibility notes

The methods distinguish the original SPAM mapping from accepted 2000 completion evidence and the 23 MIRCA2000 irrigation-share corrections. The 2015 grid midpoint and frozen 2015 geography are also distinct steps. Document both the validated results and what is not yet reproduced from raw sources.

In [ ]:
# 10 | Scientific and reproducibility documentation
readme_05 = f"""# SPAM to FABLE historical crop-area dataset ({VERSION})

**Release status:** Validated scientific candidate. Public deposit is not yet approved.

## Dataset

The main file is `data/{REQUIRED_05['candidate_csv'].name}`. It contains 52,864
country-year-product records, 34 Calculator fields and 56 canonical FABLE products.
Years: 2000, 2005, 2010, 2015 and 2020.

Country coverage depends on the SPAM release: 177, 184, 201, 203 and 179 countries,
respectively. These are separate year-specific country panels, not a fixed global panel.

## Processing and scientific sources

See `docs/METHODS.md` and `docs/PROVENANCE.md`. Raw SPAM and external mapping
inputs are not redistributed by default. Mapped 5 arc-minute layers are
recorded in the spatial lineage inventory, and may be deposited in a
separate data archive after checking size and source distribution rights. The source inventory contains hashes and
local paths used during validation, which should be replaced with a portable data
acquisition guide before public deposit.

## Data dictionary and validation

The complete dictionary is in `data/04_historical_data_dictionary.csv`.
Saved QA summaries and the new independent release audit are under `qc/`.
Notebook 03: 8 of 8 required audit gates passed.
Notebook 04: 4 scientific gates passed and all 19 numerical checks passed.

## Reproducibility status

{clean_result_05['status']}

A source-to-release clean run is a separate gate. Checking previously produced data
and replaying 2015 from saved mapped 2010 and 2020 files does not establish a
full raw-source rebuild. See `docs/REPRODUCIBILITY.md`.

## Release and license review

Creators, data and code licenses, redistribution permissions and institutional
approval must be confirmed before publication. The draft metadata is under
`reproducibility/publication_metadata.template.json`.

The canonical FABLE Calculator FINAL was not changed by this workflow.
"""
methods_05 = """# Methods

## Input years and spatial resolution

SPAM global crop data for 2000, 2005, 2010 and 2020 were standardized at native
5 arc-minute resolution. Irrigated and rainfed harvested area and physical area
were kept as separate components. The FABLE mapping uses direct and weighted
source-to-product allocations with country-year frozen weights where required.

## Year 2000 and correction provenance

The 2000 FABLE product completion includes accepted FAOSTAT and same-year
MIRCA2000 crop-magnitude evidence for selected cases. This is a magnitude-evidence
process, not a general replacement of SPAM area data. Separately, 23 approved
country-product irrigation-share corrections were restored using the archived
MIRCA2000 harmonization evidence. The corrected release conserves harvested
and physical totals for those 23 records. The record-level before and after
values are provided in the change register.

Other oilseed (OOIL) parent allocation was checked across 159 parents and seven
FABLE child products, covering 1,113 country-child allocations. The later audit
used a deterministic source-grid replay, not a preserved original intermediate
pre-aggregation export.

## Year 2015

The four primary area components were separately interpolated between mapped
2010 and 2020 grid-product endpoints. The frozen 2015 country crosswalk was
then applied. Only after spatial aggregation were total areas and harvesting
intensities calculated. The 2015 grid replay compared 24,060,400 grid-product
keys without a component or country mismatch in the validated run. The frozen
geography crosswalk itself was not regenerated from raw boundaries by Notebook 04.

## Area and intensity conventions

Areas are stored in hectares and thousand hectares. Intensities are harvested
area divided by physical area and are not capped at one. For the historical
zero-harvest and zero-physical (0/0) case, saved intensity may be zero or
missing. Validation retains either historical representation.

## Validation limits

These checks confirm a validated versioned candidate. Source-specific OOIL
allocations were replayed and compared against the saved mapped product output.
A full fresh raw-source-to-candidate rebuild is a separate release gate.
"""
provenance_05 = f"""# Scientific provenance

Notebook 01: native SPAM source standardization and 2000 OTHE source-only repair.
Notebook 02: frozen FABLE 56-product mapping and country-year weights.
Notebook 03: 2000 reconstruction audit, MIRCA magnitude and share provenance,
OOIL source replay, corrected historical candidate and independent disk QC.
Notebook 04: 2015 mapped-grid interpolation replay, country reconciliation,
full Calculator QC and versioned output.
Notebook 05: independent release audit, source-lock verification and packaging.

Historical candidate SHA-256: `{candidate_hash_05}`.
Canonical Calculator FINAL SHA-256: `{canonical_hash_05}`.

Year 2000 corrected records: 23. Other historical records remain unchanged within
approved serialization precision. The current candidate is separate from the
canonical Calculator production file.

All source input locations and hashes are recorded in
`reproducibility/04_frozen_input_SHA256_inventory.csv`.
"""
repro_05 = f"""# How to reproduce the data

## Environment

Use a Python environment with numpy, pandas, duckdb, pyarrow, pycountry,
openpyxl, nbformat, nbclient, ipykernel and IPython. The environment snapshot
and minimum dependency list are included with the bundle.

## Execution order

1. Prepare native SPAM source releases from the raw source files.
2. Freeze and validate the SPAM-to-FABLE direct and weighted mapping.
3. Rebuild mapped observed 5 arc-minute grid areas.
4. Rebuild 2000 FABLE product completion and the approved MIRCA2000 share correction.
5. Rebuild the 2015 grid midpoint with the frozen 2015 country crosswalk.
6. Assemble the final five-year historical Calculator table and compare all 34 columns.

Use a fresh isolated workspace, new kernel per notebook, saved execution logs
and source and output fingerprints for every stage. Never use the original
canonical FINAL or an existing Notebook 03 candidate as the newly generated
result. A partial 01 to 03 raw-source replay is useful evidence but is not
itself a full clean-start reproduction of the corrected historical release.

## Current independent check

The Notebook 05 reproducibility gate is: {clean_result_05['status']}.
Full proof requires the source-to-release run manifest and comparison with the
validated CSV. No clean rebuild is marked PASS merely because this notebook
itself executed successfully.

## Known workflow gap

The earlier Notebook 03 observed-grid production variants require different
product-representability inputs, and the extended 2000 country-product
completion is in the older mapping and reconstruction workflow. Assemble
and parameterize the actual production implementation for the clean runner.
A stale Notebook 03 review file must not be represented as the final source.
"""

for name, body in {
    "README.md": readme_05,
    "docs/METHODS.md": methods_05,
    "docs/PROVENANCE.md": provenance_05,
    "docs/REPRODUCIBILITY.md": repro_05,
}.items():
    p = BUNDLE_DIR / name
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(body, encoding="utf-8")

requirements_05 = """numpy>=1.26,<3
pandas>=2.1,<3
duckdb>=1.0,<2
pyarrow>=14,<25
pycountry>=22
openpyxl>=3.1
nbformat>=5.10
nbclient>=0.10
jupyter_client>=8
ipykernel>=6
ipython>=8
"""
(BUNDLE_DIR / "reproducibility" / "requirements.txt").write_text(
    requirements_05, encoding="utf-8")

versions_05 = {}
for pkg in ["numpy", "pandas", "duckdb", "pyarrow", "pycountry", "openpyxl",
            "nbformat", "nbclient", "jupyter_client", "ipykernel", "ipython"]:
    try:
        versions_05[pkg] = importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        versions_05[pkg] = "not_installed"
(BUNDLE_DIR / "reproducibility" / "environment_snapshot.json").write_text(
    json.dumps({"python": sys.version, "platform": platform.platform(),
                "packages": versions_05}, indent=2), encoding="utf-8")
print("Methodology, provenance and reproducibility notes exported.")

## 11 | Final gate table, package checksums and publication manifest

Scientific validation and public-release approval are separate. The notebook may finish successfully while full clean-start reproduction or administrative review remains pending. Every package artifact is included in a SHA-256 manifest; the checksum file itself is excluded to avoid self-reference.

In [ ]:
# 11 | Publication decision and verifiable file inventory
gates_05 = pd.DataFrame([
    {"gate": "NOTEBOOK03_SCIENTIFIC_AUDIT", "passed": True,
     "status": "PASS_8_OF_8"},
    {"gate": "NOTEBOOK04_SCIENTIFIC_VALIDATION", "passed": True,
     "status": "PASS_4_OF_4"},
    {"gate": "INDEPENDENT_FIVE_YEAR_QC", "passed": bool(rechecked_qc_05.failures.eq(0).all()),
     "status": "PASS" if rechecked_qc_05.failures.eq(0).all() else "FAIL"},
    {"gate": "HISTORICAL_IDENTITY_AND_MIRCA23", "passed": True,
     "status": "PASS_23_OF_23"},
    {"gate": "SOURCE_NOTEBOOKS_COMPLETE", "passed": source_code_complete_05,
     "status": "PASS" if source_code_complete_05 else "PENDING_FINAL_SOURCE_NOTEBOOKS"},
    {"gate": "FULL_CLEAN_START_REPRODUCIBILITY", "passed": fresh_rebuild_pass_05,
     "status": clean_result_05["status"]},
    {"gate": "METADATA_AND_INPUT_LICENSE_REVIEW", "passed": metadata_complete_05,
     "status": "PASS" if metadata_complete_05 else "PENDING_METADATA_OR_SOURCE_LICENSE"},
    {"gate": "INSTITUTIONAL_RELEASE_APPROVAL", "passed": institutional_approval_05,
     "status": "PASS" if institutional_approval_05 else "PENDING_EXPLICIT_APPROVAL"},
])
scientific_validated_05 = bool(gates_05.loc[:3, "passed"].all())
public_release_ready_05 = bool(gates_05.passed.all())
status_05 = ("READY_FOR_REVIEWED_PUBLIC_DEPOSIT" if public_release_ready_05
             else "VALIDATED_CANDIDATE_PUBLIC_RELEASE_PENDING")

# Final QC and manifest are allowed to change while this is a draft bundle.
gates_05.to_csv(BUNDLE_DIR / "qc" / "05_publication_release_gates.csv", index=False)
payload_05 = sorted(
    p for p in BUNDLE_DIR.rglob("*")
    if p.is_file() and p.name not in {"05_publication_manifest.json", "SHA256SUMS.txt"}
)
inventory_05 = [
    {"path": p.relative_to(BUNDLE_DIR).as_posix(), "bytes": p.stat().st_size,
     "sha256": sha256_05(p)}
    for p in payload_05
]
publication_manifest_05 = {
    "dataset": metadata_05["title"],
    "version": VERSION,
    "status": status_05,
    "scientific_validated": scientific_validated_05,
    "public_release_ready": public_release_ready_05,
    "historical_records": len(historical_05),
    "calculator_columns": len(CALCULATOR_05),
    "approved_MIRCA2000_corrections": 23,
    "notebook03_audit_gates": 8,
    "notebook04_scientific_gates": 4,
    "candidate_sha256": candidate_hash_05,
    "canonical_final_sha256": canonical_hash_05,
    "canonical_final_modified": False,
    "clean_start": clean_result_05,
    "release_gates": gates_05.to_dict(orient="records"),
    "bundle_files": inventory_05,
}
(BUNDLE_DIR / "05_publication_manifest.json").write_text(
    json.dumps(publication_manifest_05, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8")
checksum_lines_05 = [
    f"{sha256_05(p)}  {p.relative_to(BUNDLE_DIR).as_posix()}"
    for p in payload_05
]
checksum_lines_05.append(
    f"{sha256_05(BUNDLE_DIR / '05_publication_manifest.json')}  05_publication_manifest.json"
)
(BUNDLE_DIR / "SHA256SUMS.txt").write_text(
    "\n".join(checksum_lines_05) + "\n", encoding="utf-8")

summary_05 = pd.DataFrame([{
    "stage": "NOTEBOOK05_PUBLICATION_AUDIT",
    "scientific_gates_passed": int(gates_05.loc[:3, "passed"].sum()),
    "historical_records": len(historical_05),
    "schema_columns": len(historical_05.columns),
    "MIRCA2000_corrections": 23,
    "independent_numerical_failures": int(rechecked_qc_05.failures.sum()),
    "clean_rebuild_status": clean_result_05["status"],
    "notebook_sources_complete": source_code_complete_05,
    "metadata_complete": metadata_complete_05,
    "public_release_ready": public_release_ready_05,
    "status": status_05,
}])
summary_05.to_csv(AUDIT_DIR_05 / "05_final_publication_audit_summary.csv", index=False)
gates_05.to_csv(AUDIT_DIR_05 / "05_publication_release_gates.csv", index=False)
display(summary_05)
display(gates_05)
print("Publication bundle:", BUNDLE_DIR)
print("Release remains unpromoted. Canonical FINAL is unchanged.")

## Publication decision

If the status is `VALIDATED_CANDIDATE_PUBLIC_RELEASE_PENDING`, the notebook has completed its audit and assembled a review bundle. This is expected until a genuinely fresh reconstruction has been verified, the actual final Notebook 03 is saved, and the creators, licenses and institutional review are recorded.

If all gates eventually pass, this notebook reports `READY_FOR_REVIEWED_PUBLIC_DEPOSIT`. It never uploads to GitHub or Zenodo and never replaces the Calculator's canonical FINAL automatically. Final external deposit and production promotion require explicit approval.